# 00 — Explore MIND (MINDsmall)

Exploratory Data Analysis for the **MINDsmall** dataset (English, TSV format).

This notebook inventories file structure, row counts, schema, timestamp ranges, click-through rate, per-user / per-article distributions, missing text, and entity annotations.

In [1]:
import json
from pathlib import Path
from collections import Counter
from datetime import datetime

PROJECT_ROOT = Path(".").resolve().parent
MIND_DIR = PROJECT_ROOT / "data" / "raw" / "mind"
TRAIN_DIR = MIND_DIR / "MINDsmall_train"
DEV_DIR = MIND_DIR / "MINDsmall_dev"

NEWS_COLS = ["news_id", "category", "subcategory", "title", "abstract", "url",
             "title_entities", "abstract_entities"]
BEHAVIORS_COLS = ["impression_id", "user_id", "time", "history", "impressions"]

print(f"Project root: {PROJECT_ROOT}")
print(f"MIND dir: {MIND_DIR}")

Project root: /home/shrawani/Desktop/sem5/Information Retrieval and Extraction/assignment-1/ire_a1/recsys-ir
MIND dir: /home/shrawani/Desktop/sem5/Information Retrieval and Extraction/assignment-1/ire_a1/recsys-ir/data/raw/mind


## 1. File Listing and Sizes

In [2]:
import subprocess
result = subprocess.run(["ls", "-lhR", str(MIND_DIR)], capture_output=True, text=True)
print(result.stdout)

/home/shrawani/Desktop/sem5/Information Retrieval and Extraction/assignment-1/ire_a1/recsys-ir/data/raw/mind:
total 81M
drwxrwxr-x 2 shrawani shrawani 4.0K Aug 14 18:19 MINDsmall_dev
-rw-rw-r-- 1 shrawani shrawani  30M Aug 14 18:19 MINDsmall_dev.zip
drwxrwxr-x 2 shrawani shrawani 4.0K Aug 14 18:04 MINDsmall_train
-rw-rw-r-- 1 shrawani shrawani  51M Aug 14 18:14 MINDsmall_train.zip

/home/shrawani/Desktop/sem5/Information Retrieval and Extraction/assignment-1/ire_a1/recsys-ir/data/raw/mind/MINDsmall_dev:
total 95M
-rw-rw-r-- 1 shrawani shrawani   41M Aug 14 18:19 behaviors.tsv
-rw-rw-r-- 1 shrawani shrawani   21M Aug 14 18:19 entity_embedding.vec
-rw-rw-r-- 1 shrawani shrawani   32M Aug 14 18:19 news.tsv
-rw-rw-r-- 1 shrawani shrawani 1021K Aug 14 18:19 relation_embedding.vec

/home/shrawani/Desktop/sem5/Information Retrieval and Extraction/assignment-1/ire_a1/recsys-ir/data/raw/mind/MINDsmall_train:
total 153M
-rw-rw-r-- 1 shrawani shrawani   88M Aug 14 18:04 behaviors.tsv
-rw-rw-r-- 1

## 2. Row Counts

In [3]:
for split_name, split_dir in [("train", TRAIN_DIR), ("dev", DEV_DIR)]:
    print(f"\n--- {split_name} ---")
    for f in sorted(split_dir.glob("*")):
        if f.is_file():
            with open(f) as fh:
                count = sum(1 for _ in fh)
            print(f"  {f.name}: {count:,} rows")


--- train ---
  behaviors.tsv: 156,965 rows
  entity_embedding.vec: 26,904 rows
  news.tsv: 51,282 rows
  relation_embedding.vec: 1,091 rows

--- dev ---
  behaviors.tsv: 73,152 rows
  entity_embedding.vec: 22,893 rows
  news.tsv: 42,416 rows
  relation_embedding.vec: 1,091 rows


## 3. Schema and Sample Rows

In [4]:
def read_tsv(path, cols):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= len(cols):
                rows.append(dict(zip(cols, parts[:len(cols)])))
    return rows

for split_name, split_dir in [("train", TRAIN_DIR), ("dev", DEV_DIR)]:
    print(f"\n{'='*60}")
    print(f"  {split_name}")
    print(f"{'='*60}")
    
    news = read_tsv(split_dir / "news.tsv", NEWS_COLS)
    print(f"\nNews columns: {NEWS_COLS}")
    print(f"News rows: {len(news):,}")
    for i, row in enumerate(news[:5]):
        print(f"  [{i}] {row['news_id']} | {row['category']} | {row['title'][:70]}…")
    
    behaviors = read_tsv(split_dir / "behaviors.tsv", BEHAVIORS_COLS)
    print(f"\nBehaviors columns: {BEHAVIORS_COLS}")
    print(f"Behaviors rows: {len(behaviors):,}")
    for i, row in enumerate(behaviors[:5]):
        n_hist = len(row.get('history', '').split()) if row.get('history') else 0
        n_imp = len(row.get('impressions', '').split())
        print(f"  [{i}] imp={row['impression_id']} user={row['user_id']} time={row['time']} hist={n_hist} imp={n_imp}")


  train

News columns: ['news_id', 'category', 'subcategory', 'title', 'abstract', 'url', 'title_entities', 'abstract_entities']
News rows: 51,282
  [0] N55528 | lifestyle | The Brands Queen Elizabeth, Prince Charles, and Prince Philip Swear By…
  [1] N19639 | health | 50 Worst Habits For Belly Fat…
  [2] N61837 | news | The Cost of Trump's Aid Freeze in the Trenches of Ukraine's War…
  [3] N53526 | health | I Was An NBA Wife. Here's How It Affected My Mental Health.…
  [4] N38324 | health | How to Get Rid of Skin Tags, According to a Dermatologist…

Behaviors columns: ['impression_id', 'user_id', 'time', 'history', 'impressions']
Behaviors rows: 156,965
  [0] imp=1 user=U13740 time=11/11/2019 9:05:58 AM hist=9 imp=2
  [1] imp=2 user=U91836 time=11/12/2019 6:11:30 PM hist=82 imp=11
  [2] imp=3 user=U73700 time=11/14/2019 7:01:48 AM hist=16 imp=36
  [3] imp=4 user=U34670 time=11/11/2019 5:28:05 AM hist=10 imp=4
  [4] imp=5 user=U8125 time=11/12/2019 4:11:21 PM hist=4 imp=69

  dev

New

## 4. Timestamp Range (Impressions)

In [5]:
for split_name, split_dir in [("train", TRAIN_DIR), ("dev", DEV_DIR)]:
    behaviors = read_tsv(split_dir / "behaviors.tsv", BEHAVIORS_COLS)
    times = []
    for b in behaviors:
        try:
            t = datetime.strptime(b["time"], "%m/%d/%Y %I:%M:%S %p")
            times.append(t)
        except ValueError:
            pass
    print(f"{split_name}: {min(times).isoformat()} → {max(times).isoformat()}")

train: 2019-11-09T00:00:19 → 2019-11-14T23:59:13
dev: 2019-11-15T00:00:01 → 2019-11-15T23:58:03


## 5. Click-Through Rate

In [6]:
for split_name, split_dir in [("train", TRAIN_DIR), ("dev", DEV_DIR)]:
    behaviors = read_tsv(split_dir / "behaviors.tsv", BEHAVIORS_COLS)
    total, pos = 0, 0
    for b in behaviors:
        for imp in b.get("impressions", "").split():
            if "-" in imp:
                _, label = imp.rsplit("-", 1)
                total += 1
                if label == "1":
                    pos += 1
    ctr = pos / total if total else 0
    print(f"{split_name}: {pos:,} clicks / {total:,} pairs = CTR {ctr:.4f} ({ctr*100:.2f}%)")

train: 236,344 clicks / 5,843,444 pairs = CTR 0.0404 (4.04%)
dev: 111,383 clicks / 2,740,998 pairs = CTR 0.0406 (4.06%)


## 6. Clicks per User Distribution

In [7]:
for split_name, split_dir in [("train", TRAIN_DIR), ("dev", DEV_DIR)]:
    behaviors = read_tsv(split_dir / "behaviors.tsv", BEHAVIORS_COLS)
    user_clicks = Counter()
    for b in behaviors:
        uid = b["user_id"]
        for imp in b.get("impressions", "").split():
            if "-" in imp:
                _, label = imp.rsplit("-", 1)
                if label == "1":
                    user_clicks[uid] += 1
    
    vals = sorted(user_clicks.values())
    n = len(vals)
    print(f"\n{split_name} — {n:,} users with ≥1 click")
    print(f"  min={vals[0]}, max={vals[-1]}, mean={sum(vals)/n:.2f}")
    print(f"  median={vals[n//2]}, p75={vals[int(n*0.75)]}, p90={vals[int(n*0.90)]}")
    print(f"  p95={vals[int(n*0.95)]}, p99={vals[int(n*0.99)]}")


train — 50,000 users with ≥1 click
  min=1, max=129, mean=4.73
  median=3, p75=6, p90=10
  p95=15, p99=28

dev — 50,000 users with ≥1 click
  min=1, max=33, mean=2.23
  median=1, p75=3, p90=4
  p95=6, p99=11


## 7. Impressions per Article Distribution

In [8]:
for split_name, split_dir in [("train", TRAIN_DIR), ("dev", DEV_DIR)]:
    behaviors = read_tsv(split_dir / "behaviors.tsv", BEHAVIORS_COLS)
    article_counts = Counter()
    for b in behaviors:
        for imp in b.get("impressions", "").split():
            if "-" in imp:
                nid = imp.rsplit("-", 1)[0]
                article_counts[nid] += 1
    
    vals = sorted(article_counts.values(), reverse=True)
    n = len(vals)
    print(f"\n{split_name} — {n:,} articles appeared in impressions")
    print(f"  min={vals[-1]}, max={vals[0]}, mean={sum(vals)/n:.2f}, median={vals[n//2]}")
    print(f"  Top 10 impression counts: {vals[:10]}")


train — 20,288 articles appeared in impressions
  min=1, max=23037, mean=288.02, median=7
  Top 10 impression counts: [23037, 19242, 19106, 18702, 18315, 18101, 16869, 16267, 16022, 15551]

dev — 5,369 articles appeared in impressions
  min=1, max=47285, mean=510.52, median=11
  Top 10 impression counts: [47285, 39840, 36205, 36152, 34151, 32861, 32769, 32046, 29153, 27204]


## 8. Missing Abstract/Body Text

In [9]:
for split_name, split_dir in [("train", TRAIN_DIR), ("dev", DEV_DIR)]:
    news = read_tsv(split_dir / "news.tsv", NEWS_COLS)
    n = len(news)
    missing_abstract = sum(1 for a in news if not a.get("abstract", "").strip())
    missing_title = sum(1 for a in news if not a.get("title", "").strip())
    print(f"\n{split_name} ({n:,} articles):")
    print(f"  Missing abstract: {missing_abstract} ({missing_abstract/n*100:.1f}%)")
    print(f"  Missing title: {missing_title} ({missing_title/n*100:.1f}%)")
    print(f"  Note: MIND does not provide article body text in TSV; body available via URL only")


train (51,282 articles):
  Missing abstract: 2666 (5.2%)
  Missing title: 0 (0.0%)
  Note: MIND does not provide article body text in TSV; body available via URL only

dev (42,416 articles):
  Missing abstract: 2021 (4.8%)
  Missing title: 0 (0.0%)
  Note: MIND does not provide article body text in TSV; body available via URL only


## 9. Entity Annotation Format (MIND-specific)

In [10]:
news = read_tsv(TRAIN_DIR / "news.tsv", NEWS_COLS)

print("Entity annotation format:")
print("  Each article has title_entities and abstract_entities columns")
print("  Format: JSON list of {Label, Type, WikidataId, Confidence, OccurrenceOffsets, SurfaceForms}")
print()

# Show 3 examples
for a in news[:20]:
    te = a.get("title_entities", "")
    if te and te != "[]":
        parsed = json.loads(te)
        print(f"  {a['news_id']}: {len(parsed)} title entities")
        for e in parsed[:2]:
            print(f"    WikidataId={e['WikidataId']}, Label={e['Label']}, Type={e['Type']}, Conf={e['Confidence']}")
        break

# Entity embedding format
print("\nEntity embedding format:")
with open(TRAIN_DIR / "entity_embedding.vec") as f:
    line = f.readline().strip()
parts = line.split("\t")
print(f"  WikiData ID: {parts[0]}")
print(f"  Embedding dim: {len(parts) - 1}")
print(f"  File: entity_embedding.vec (train: 26,904 entities, dev: 22,893)")

with open(TRAIN_DIR / "relation_embedding.vec") as f:
    line = f.readline().strip()
parts = line.split("\t")
print(f"  Relation embedding dim: {len(parts) - 1}")
print(f"  File: relation_embedding.vec (train/dev: 1,091 relations each)")

Entity annotation format:
  Each article has title_entities and abstract_entities columns
  Format: JSON list of {Label, Type, WikidataId, Confidence, OccurrenceOffsets, SurfaceForms}

  N55528: 3 title entities
    WikidataId=Q80976, Label=Prince Philip, Duke of Edinburgh, Type=P, Conf=1.0
    WikidataId=Q43274, Label=Charles, Prince of Wales, Type=P, Conf=1.0

Entity embedding format:
  WikiData ID: Q41
  Embedding dim: 100
  File: entity_embedding.vec (train: 26,904 entities, dev: 22,893)
  Relation embedding dim: 100
  File: relation_embedding.vec (train/dev: 1,091 relations each)
